In [3]:
# =========================
# FINAL HYBRID + SOC SYSTEM (CORRECTED)
# =========================

import pandas as pd
import numpy as np
import time
import re
import random
from collections import Counter

# =========================
# LOAD DATA
# =========================
train_df = pd.read_csv("UNSW_NB15_grouped_training_class.csv")
test_df  = pd.read_csv("UNSW_NB15_grouped_testing_class.csv")

train_df["target"] = train_df["4_class"].apply(lambda x: 0 if x == "Benign" else 1)
test_df["target"]  = test_df["4_class"].apply(lambda x: 0 if x == "Benign" else 1)

# =========================
# PREPROCESSING
# =========================
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

drop_cols = ["id","attack_cat","label","4_class","target"]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df["target"]

X_test  = test_df.drop(columns=drop_cols)
y_test  = test_df["target"]

cat_cols = ["proto","service","state"]

preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)],
    remainder="passthrough"
)

X_train = preprocessor.fit_transform(X_train)
X_test  = preprocessor.transform(X_test)

scaler = StandardScaler(with_mean=False)
X_train = scaler.fit_transform(X_train).toarray()
X_test  = scaler.transform(X_test).toarray()

# =========================
# SEQUENCES
# =========================
def create_sequences(X, y, seq_len):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_len):
        X_seq.append(X[i:i+seq_len])
        y_seq.append(1 if np.any(y.iloc[i:i+seq_len].values == 1) else 0)
    return np.array(X_seq), np.array(y_seq)

seq_len = 10
X_train_seq, y_train_seq = create_sequences(X_train, y_train, seq_len)
X_test_seq, y_test_seq   = create_sequences(X_test, y_test, seq_len)

# =========================
# MODEL
# =========================
from tensorflow.keras.layers import Input, Dense, LayerNormalization, MultiHeadAttention, Dropout, GlobalAveragePooling1D
from tensorflow.keras.models import Model

def transformer_block(inputs):
    x = MultiHeadAttention(key_dim=64, num_heads=4)(inputs, inputs)
    x = LayerNormalization()(x + inputs)
    ffn = Dense(128, activation="relu")(x)
    ffn = Dense(inputs.shape[-1])(ffn)
    return LayerNormalization()(x + ffn)

inputs = Input(shape=(seq_len, X_train_seq.shape[2]))
x = transformer_block(inputs)
x = transformer_block(x)
x = GlobalAveragePooling1D()(x)
x = Dense(64, activation="relu")(x)
x = Dropout(0.3)(x)
outputs = Dense(1, activation="sigmoid")(x)

model = Model(inputs, outputs)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# =========================
# TRAIN
# =========================
start_train = time.time()
model.fit(X_train_seq, y_train_seq, epochs=3, batch_size=64)
training_time = time.time() - start_train
print("\n⏱ TRAINING TIME:", round(training_time,2), "sec")

# =========================
# RULES
# =========================
def load_rules(filepath):
    rules=[]
    try:
        with open(filepath,"r") as f:
            for line in f:
                line=line.strip()
                if not line or line.startswith("#"): continue
                if line.endswith(","): line=line[:-1]
                try:
                    name,pattern,cat=eval(line)
                    rules.append((name,re.compile(pattern,re.I),cat))
                except:
                    continue
    except:
        open(filepath,"w").close()
    return rules

def rule_exists(rules, pattern):
    return any(p.pattern == pattern for _,p,_ in rules)

def append_rule(filepath, name, pattern, cat):
    with open(filepath,"a") as f:
        f.write(f'("{name}", r"{pattern}", "{cat}"),\n')

rules = load_rules("rules.txt")
print("Loaded Rules:", len(rules))

# =========================
# HYBRID DETECTION
# =========================
texts=[f"{r.proto} {r.service} {r.state} {r.attack_cat}" for _,r in test_df.iterrows()]

start_detect = time.time()
preds = model.predict(X_test_seq, verbose=0)

results=[]
for i in range(len(X_test_seq)):
    row=test_df.iloc[i+seq_len-1]
    text=texts[i+seq_len-1]

    found=False
    for name,p,cat in rules:
        if p.search(text):
            results.append((1,cat))
            found=True
            break

    if not found:
        prob=preds[i][0]
        if prob>0.9:
            attack=row["attack_cat"].lower()
            if attack!="normal":
                pattern=f"\\b{row['proto']}.*{attack}\\b"
                if not rule_exists(rules,pattern):
                    append_rule("rules.txt",f"dl_{attack}_{i}",pattern,attack)
                    rules.append((f"dl_{attack}_{i}",re.compile(pattern,re.I),attack))
            results.append((1,attack))
        else:
            results.append((0,"normal"))

detect_time=(time.time()-start_detect)/len(X_test_seq)
print("⏱ DETECTION TIME/SEQ:",detect_time)

# =========================
# METRICS
# =========================
from sklearn.metrics import confusion_matrix,precision_score,recall_score,f1_score,accuracy_score

y_pred=[x[0] for x in results]

cm=confusion_matrix(y_test_seq,y_pred)
tn,fp,fn,tp=cm.ravel()

print("\nCONFUSION MATRIX:\n",cm)
print("\nTP:",tp,"TN:",tn,"FP:",fp,"FN:",fn)

print("Accuracy:",accuracy_score(y_test_seq,y_pred))
print("Precision:",precision_score(y_test_seq,y_pred))
print("Recall:",recall_score(y_test_seq,y_pred))
print("F1:",f1_score(y_test_seq,y_pred))

# =========================
# SOC OUTPUT
# =========================
def random_ip():
    return ".".join(str(random.randint(1,255)) for _ in range(4))

logs=[]
for i in range(100):
    row=test_df.iloc[i]
    logs.append({
        "src":random_ip(),
        "dst":random_ip(),
        "proto":row["proto"],
        "service":row["service"],
        "attack":results[i][1]
    })

print("\n🧠 SUMMARY:")
for l in logs[:3]:
    if l["attack"]=="normal":
        print(f"Normal traffic from {l['src']} → {l['dst']} via {l['proto']}/{l['service']}")
    else:
        print(f"{l['attack']} attack from {l['src']} → {l['dst']} via {l['proto']}/{l['service']}")

attacks=[l for l in logs if l["attack"]!="normal"]

if len(attacks)==0:
    print("\n✔ No attacks detected — system normal")
    print("🚨 THREAT LEVEL: NONE")
    print("🛡️ No action needed")
else:
    src=Counter([l["src"] for l in attacks]).most_common(1)[0][0]
    dst=Counter([l["dst"] for l in attacks]).most_common(1)[0][0]
    atype=Counter([l["attack"] for l in attacks]).most_common(1)[0][0]

    print("\n🔍 EXPLAINABILITY REPORT:")
    print("WHO:",src)
    print("WHERE:",dst)
    print("WHAT:",atype)
    print("HOW:",attacks[0]["proto"],"using",attacks[0]["service"])

    level="HIGH" if len(attacks)>20 else "LOW"
    print("\n🚨 THREAT LEVEL:",level)

    print("\n🧠 WHY ANALYSIS:")
    print(f"- {atype} is dominant attack")
    print(f"- {dst} heavily targeted")
    print(f"- protocol {attacks[0]['proto']} exploited")

    print("\n🛡️ RECOMMENDED ACTIONS:")
    print(f"- Block IP {src}")
    print("- Monitor traffic")
    print("- Apply firewall rules")

Epoch 1/3
2740/2740 ━━━━━━━━━━━━━━━━━━━━ 206s 74ms/step - accuracy: 0.9913 - loss: 0.0255
Epoch 2/3
2740/2740 ━━━━━━━━━━━━━━━━━━━━ 132s 48ms/step - accuracy: 0.9939 - loss: 0.0182
Epoch 3/3
2740/2740 ━━━━━━━━━━━━━━━━━━━━ 128s 47ms/step - accuracy: 0.9948 - loss: 0.0160

⏱ TRAINING TIME: 467.75 sec
Loaded Rules: 730
⏱ DETECTION TIME/SEQ: 0.00045281135289575956

CONFUSION MATRIX:
 [[36470   502]
 [   15 45335]]

TP: 45335 TN: 36470 FP: 502 FN: 15
Accuracy: 0.9937197832900075
Precision: 0.9890481488753627
Recall: 0.9996692392502756
F1: 0.9943303321745425

🧠 SUMMARY:
Normal traffic from 154.46.34.62 → 179.130.101.37 via udp/-
Normal traffic from 50.251.216.107 → 70.4.112.137 via udp/-
Normal traffic from 205.35.129.70 → 11.42.60.2 via udp/-

✔ No attacks detected — system normal
🚨 THREAT LEVEL: NONE
🛡️ No action needed


In [3]:
# =========================
# FINAL SOC SYSTEM (FULL FIXED + OPTIMIZED)
# =========================

import pandas as pd
import numpy as np
import time
import re
import random
from collections import Counter, defaultdict

# =========================
# LOAD DATA
# =========================
train_df = pd.read_csv("UNSW_NB15_grouped_training_class.csv")
test_df  = pd.read_csv("UNSW_NB15_grouped_testing_class.csv")

train_df["target"] = train_df["4_class"].apply(lambda x: 0 if x == "Benign" else 1)
test_df["target"]  = test_df["4_class"].apply(lambda x: 0 if x == "Benign" else 1)

# =========================
# PREPROCESSING
# =========================
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

drop_cols = ["id","attack_cat","label","4_class","target"]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df["target"]

X_test  = test_df.drop(columns=drop_cols)
y_test  = test_df["target"]

cat_cols = ["proto","service","state"]

preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)],
    remainder="passthrough"
)

X_train = preprocessor.fit_transform(X_train)
X_test  = preprocessor.transform(X_test)

scaler = StandardScaler(with_mean=False)
X_train = scaler.fit_transform(X_train).toarray()
X_test  = scaler.transform(X_test).toarray()

# =========================
# SEQUENCES
# =========================
def create_sequences(X, y, seq_len):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_len):
        X_seq.append(X[i:i+seq_len])
        y_seq.append(1 if np.any(y.iloc[i:i+seq_len].values == 1) else 0)
    return np.array(X_seq), np.array(y_seq)

seq_len = 10
X_train_seq, y_train_seq = create_sequences(X_train, y_train, seq_len)
X_test_seq, y_test_seq   = create_sequences(X_test, y_test, seq_len)

# =========================
# MODEL
# =========================
from tensorflow.keras.layers import Input, Dense, LayerNormalization, MultiHeadAttention, Dropout, GlobalAveragePooling1D
from tensorflow.keras.models import Model

def transformer_block(inputs):
    x = MultiHeadAttention(key_dim=64, num_heads=4)(inputs, inputs)
    x = LayerNormalization()(x + inputs)
    ffn = Dense(128, activation="relu")(x)
    ffn = Dense(inputs.shape[-1])(ffn)
    return LayerNormalization()(x + ffn)

inputs = Input(shape=(seq_len, X_train_seq.shape[2]))
x = transformer_block(inputs)
x = transformer_block(x)
x = GlobalAveragePooling1D()(x)
x = Dense(64, activation="relu")(x)
x = Dropout(0.3)(x)
outputs = Dense(1, activation="sigmoid")(x)

model = Model(inputs, outputs)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# =========================
# TRAIN
# =========================
start_train = time.time()
model.fit(X_train_seq, y_train_seq, epochs=3, batch_size=64)
training_time = time.time() - start_train
print("\n⏱ TRAINING TIME:", round(training_time,2), "sec")

# =========================
# RULE SYSTEM
# =========================
IMPORTANT_PROTOCOLS = ["tcp", "udp", "icmp"]
MAX_RULES_PER_ATTACK = 5

rule_stats = defaultdict(lambda: {"count":0, "confidence":0})

def load_rules(filepath):
    rules=[]
    try:
        with open(filepath,"r") as f:
            for line in f:
                line=line.strip()
                if not line or line.startswith("#"): continue
                if line.endswith(","): line=line[:-1]
                try:
                    name,pattern,cat=eval(line)
                    rules.append((name,re.compile(pattern,re.I),cat))
                except:
                    continue
    except:
        open(filepath,"w").close()
    return rules

def rule_exists(rules, pattern):
    return any(p.pattern == pattern for _,p,_ in rules)

def append_rule(filepath, name, pattern, cat):
    with open(filepath,"a") as f:
        f.write(f'("{name}", r"{pattern}", "{cat}"),\n')

rules = load_rules("rules.txt")
print("Loaded Rules:", len(rules))

attack_rule_counter = Counter()

# =========================
# DETECTION
# =========================
texts=[f"{r.proto} {r.service} {r.state} {r.attack_cat}" for _,r in test_df.iterrows()]

start_detect = time.time()
preds = model.predict(X_test_seq, verbose=0)

results=[]

for i in range(len(X_test_seq)):
    row=test_df.iloc[i+seq_len-1]
    text=texts[i+seq_len-1]

    matched=False

    # RULE detection
    for name,p,cat in rules:
        if p.search(text):
            results.append((1,cat,"RULE"))
            rule_stats[name]["count"] += 1
            matched=True
            break

    # DL detection
    if not matched:
        prob=preds[i][0]

        if prob>0.9:
            attack=row["attack_cat"].lower()

            if attack!="normal" and row["proto"] in IMPORTANT_PROTOCOLS:
                if attack_rule_counter[attack] < MAX_RULES_PER_ATTACK:
                    pattern=f"\\b{row['proto']}.*{attack}\\b"

                    if not rule_exists(rules,pattern):
                        name=f"dl_{attack}_{attack_rule_counter[attack]}"
                        append_rule("rules.txt",name,pattern,attack)
                        rules.append((name,re.compile(pattern,re.I),attack))
                        attack_rule_counter[attack]+=1

                        rule_stats[name]["confidence"] = prob
                        print("\n🆕 RULE ADDED:", name)

            results.append((1,attack,"DL"))
        else:
            results.append((0,"normal","DL"))

detect_time=(time.time()-start_detect)/len(X_test_seq)
print("⏱ DETECTION TIME/SEQ:",detect_time)

# =========================
# METRICS
# =========================
from sklearn.metrics import confusion_matrix,precision_score,recall_score,f1_score,accuracy_score

y_pred=[x[0] for x in results]

cm=confusion_matrix(y_test_seq,y_pred)
tn,fp,fn,tp=cm.ravel()

print("\nCONFUSION MATRIX:\n",cm)
print("\nTP:",tp,"TN:",tn,"FP:",fp,"FN:",fn)

print("Accuracy:",accuracy_score(y_test_seq,y_pred))
print("Precision:",precision_score(y_test_seq,y_pred))
print("Recall:",recall_score(y_test_seq,y_pred))
print("F1:",f1_score(y_test_seq,y_pred))

print("\nTotal Attacks Detected:", sum(y_pred))

# =========================
# SOC OUTPUT (FIXED)
# =========================
def random_ip():
    return ".".join(str(random.randint(1,255)) for _ in range(4))

logs=[]
for i in range(len(results)):
    row=test_df.iloc[i]
    logs.append({
        "src":random_ip(),
        "dst":random_ip(),
        "proto":row["proto"],
        "service":row["service"],
        "attack":results[i][1],
        "source":results[i][2]
    })

print("\n🧠 SUMMARY:")

attack_logs=[l for l in logs if l["attack"]!="normal"]

if len(attack_logs)>0:
    for l in attack_logs[:3]:
        print(f"{l['attack']} attack from {l['src']} → {l['dst']} via {l['proto']}/{l['service']} (Detected by {l['source']})")
else:
    for l in logs[:3]:
        print(f"Normal traffic from {l['src']} → {l['dst']} via {l['proto']}/{l['service']} (Detected by {l['source']})")

# =========================
# EXPLAINABILITY
# =========================
if len(attack_logs)==0:
    print("\n✔ No attacks detected — system normal")
    print("🚨 THREAT LEVEL: NONE")
    print("🛡️ No action needed")

else:
    src=Counter([l["src"] for l in attack_logs]).most_common(1)[0][0]
    dst=Counter([l["dst"] for l in attack_logs]).most_common(1)[0][0]
    atype=Counter([l["attack"] for l in attack_logs]).most_common(1)[0][0]
    source=Counter([l["source"] for l in attack_logs]).most_common(1)[0][0]

    print("\n🔍 EXPLAINABILITY REPORT:")
    print("WHO:",src)
    print("WHERE:",dst)
    print("WHAT:",atype)
    print("DETECTED BY:",source)

    print("\n🚨 THREAT LEVEL:", "HIGH" if len(attack_logs)>50 else "LOW")

    print("\n🧠 WHY ANALYSIS:")
    print(f"- {atype} detected frequently")
    print(f"- {dst} targeted multiple times")
    print(f"- Detection mainly via {source}")

    print("\n🛡️ RECOMMENDED ACTIONS:")
    print(f"- Block IP {src}")
    print("- Monitor traffic")
    print("- Apply firewall rules")

Epoch 1/3
2740/2740 ━━━━━━━━━━━━━━━━━━━━ 104s 36ms/step - accuracy: 0.9916 - loss: 0.0242
Epoch 2/3
2740/2740 ━━━━━━━━━━━━━━━━━━━━ 97s 35ms/step - accuracy: 0.9930 - loss: 0.0207
Epoch 3/3
2740/2740 ━━━━━━━━━━━━━━━━━━━━ 96s 35ms/step - accuracy: 0.9937 - loss: 0.0177

⏱ TRAINING TIME: 300.85 sec
Loaded Rules: 122

🆕 RULE ADDED: dl_exploits_0

🆕 RULE ADDED: dl_exploits_1

🆕 RULE ADDED: dl_reconnaissance_0

🆕 RULE ADDED: dl_reconnaissance_1

🆕 RULE ADDED: dl_fuzzers_0

🆕 RULE ADDED: dl_dos_0

🆕 RULE ADDED: dl_fuzzers_1

🆕 RULE ADDED: dl_worms_0

🆕 RULE ADDED: dl_shellcode_0

🆕 RULE ADDED: dl_dos_1

🆕 RULE ADDED: dl_shellcode_1

🆕 RULE ADDED: dl_worms_1

🆕 RULE ADDED: dl_generic_0

🆕 RULE ADDED: dl_generic_1

🆕 RULE ADDED: dl_analysis_0
⏱ DETECTION TIME/SEQ: 0.000396552581932613

CONFUSION MATRIX:
 [[36766   206]
 [   21 45329]]

TP: 45329 TN: 36766 FP: 206 FN: 21
Accuracy: 0.9972425354097325
Precision: 0.9954760074667838
Recall: 0.9995369349503859
F1: 0.9975023381196017

Total Attacks De

In [ ]:
# =========================
# FINAL SOC SYSTEM (ULTIMATE VERSION)
# =========================

import pandas as pd
import numpy as np
import time
import re
import random
from collections import Counter, defaultdict

# =========================
# LOAD DATA
# =========================
train_df = pd.read_csv("UNSW_NB15_grouped_training_class.csv")
test_df  = pd.read_csv("UNSW_NB15_grouped_testing_class.csv")

train_df["target"] = train_df["4_class"].apply(lambda x: 0 if x == "Benign" else 1)
test_df["target"]  = test_df["4_class"].apply(lambda x: 0 if x == "Benign" else 1)

# =========================
# PREPROCESSING
# =========================
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

drop_cols = ["id","attack_cat","label","4_class","target"]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df["target"]

X_test  = test_df.drop(columns=drop_cols)
y_test  = test_df["target"]

cat_cols = ["proto","service","state"]

preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)],
    remainder="passthrough"
)

X_train = preprocessor.fit_transform(X_train)
X_test  = preprocessor.transform(X_test)

scaler = StandardScaler(with_mean=False)
X_train = scaler.fit_transform(X_train).toarray()
X_test  = scaler.transform(X_test).toarray()

# =========================
# SEQUENCES
# =========================
def create_sequences(X, y, seq_len):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_len):
        X_seq.append(X[i:i+seq_len])
        y_seq.append(1 if np.any(y.iloc[i:i+seq_len].values == 1) else 0)
    return np.array(X_seq), np.array(y_seq)

seq_len = 10
X_train_seq, y_train_seq = create_sequences(X_train, y_train, seq_len)
X_test_seq, y_test_seq   = create_sequences(X_test, y_test, seq_len)

# =========================
# MODEL
# =========================
from tensorflow.keras.layers import Input, Dense, LayerNormalization, MultiHeadAttention, Dropout, GlobalAveragePooling1D
from tensorflow.keras.models import Model

def transformer_block(inputs):
    x = MultiHeadAttention(key_dim=64, num_heads=4)(inputs, inputs)
    x = LayerNormalization()(x + inputs)
    ffn = Dense(128, activation="relu")(x)
    ffn = Dense(inputs.shape[-1])(ffn)
    return LayerNormalization()(x + ffn)

inputs = Input(shape=(seq_len, X_train_seq.shape[2]))
x = transformer_block(inputs)
x = transformer_block(x)
x = GlobalAveragePooling1D()(x)
x = Dense(64, activation="relu")(x)
x = Dropout(0.3)(x)
outputs = Dense(1, activation="sigmoid")(x)

model = Model(inputs, outputs)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# =========================
# TRAIN
# =========================
start_train = time.time()
model.fit(X_train_seq, y_train_seq, epochs=3, batch_size=64)
training_time = time.time() - start_train
print("\n⏱ TRAINING TIME:", round(training_time,2), "sec")

# =========================
# RULE SYSTEM
# =========================
IMPORTANT_PROTOCOLS = ["tcp", "udp", "icmp"]
MAX_RULES_PER_ATTACK = 5

rule_stats = defaultdict(lambda: {"count":0, "confidence":0})

def load_rules(filepath):
    rules=[]
    try:
        with open(filepath,"r") as f:
            for line in f:
                line=line.strip()
                if not line or line.startswith("#"): continue
                if line.endswith(","): line=line[:-1]
                try:
                    name,pattern,cat=eval(line)
                    rules.append((name,re.compile(pattern,re.I),cat))
                except:
                    continue
    except:
        open(filepath,"w").close()
    return rules

def rule_exists(rules, pattern):
    return any(p.pattern == pattern for _,p,_ in rules)

def append_rule(filepath, name, pattern, cat):
    with open(filepath,"a") as f:
        f.write(f'("{name}", r"{pattern}", "{cat}"),\n')

rules = load_rules("rules.txt")
print("Loaded Rules:", len(rules))

attack_rule_counter = Counter()

# =========================
# DETECTION
# =========================
texts=[f"{r.proto} {r.service} {r.state} {r.attack_cat}" for _,r in test_df.iterrows()]

start_detect = time.time()
preds = model.predict(X_test_seq, verbose=0)

results=[]

for i in range(len(X_test_seq)):
    row=test_df.iloc[i+seq_len-1]
    text=texts[i+seq_len-1]

    matched=False

    # RULE detection
    for name,p,cat in rules:
        if p.search(text):
            results.append((1,cat,"RULE"))
            rule_stats[name]["count"] += 1
            matched=True
            break

    # DL detection
    if not matched:
        prob=preds[i][0]

        if prob>0.9:
            attack=row["attack_cat"].lower()

            if attack!="normal" and row["proto"] in IMPORTANT_PROTOCOLS:
                if attack_rule_counter[attack] < MAX_RULES_PER_ATTACK:
                    pattern=f"\\b{row['proto']}.*{attack}\\b"

                    if not rule_exists(rules,pattern):
                        name=f"dl_{attack}_{attack_rule_counter[attack]}"
                        append_rule("rules.txt",name,pattern,attack)
                        rules.append((name,re.compile(pattern,re.I),attack))
                        attack_rule_counter[attack]+=1

                        rule_stats[name]["confidence"] = prob
                        print("\n🆕 RULE ADDED:", name)

            results.append((1,attack,"DL"))
        else:
            results.append((0,"normal","DL"))

detect_time=(time.time()-start_detect)/len(X_test_seq)
print("⏱ DETECTION TIME/SEQ:",detect_time)

# =========================
# RULE OPTIMIZER
# =========================
print("\n⚙️ OPTIMIZING RULES...")

# Remove weak rules
rules = [
    r for r in rules
    if rule_stats[r[0]]["count"] > 5 or rule_stats[r[0]]["confidence"] > 0.9
]

# Rank rules
ranked_rules = sorted(
    rules,
    key=lambda r: (rule_stats[r[0]]["count"], rule_stats[r[0]]["confidence"]),
    reverse=True
)

# Generalize rules
def optimize_rules(rules):
    grouped = defaultdict(list)

    for name, pattern, cat in rules:
        proto_match = re.search(r"\\b(\w+)", pattern.pattern)
        if proto_match:
            proto = proto_match.group(1)
            grouped[(proto, cat)].append(pattern.pattern)

    optimized = []

    for (proto, cat), patterns in grouped.items():
        optimized.append((f"opt_{proto}_{cat}", f"\\b{proto}.*{cat}\\b", cat))

    return optimized

optimized_rules = optimize_rules(ranked_rules)

print("\n🔥 OPTIMIZED RULES:")
for r in optimized_rules[:10]:
    print(r)

# =========================
# METRICS
# =========================
from sklearn.metrics import confusion_matrix,precision_score,recall_score,f1_score,accuracy_score

y_pred=[x[0] for x in results]

cm=confusion_matrix(y_test_seq,y_pred)
tn,fp,fn,tp=cm.ravel()

print("\nCONFUSION MATRIX:\n",cm)
print("\nTP:",tp,"TN:",tn,"FP:",fp,"FN:",fn)

print("Accuracy:",accuracy_score(y_test_seq,y_pred))
print("Precision:",precision_score(y_test_seq,y_pred))
print("Recall:",recall_score(y_test_seq,y_pred))
print("F1:",f1_score(y_test_seq,y_pred))

print("\nTotal Attacks Detected:", sum(y_pred))

# =========================
# SOC OUTPUT
# =========================
def random_ip():
    return ".".join(str(random.randint(1,255)) for _ in range(4))

logs=[]
for i in range(len(results)):
    row=test_df.iloc[i]
    logs.append({
        "src":random_ip(),
        "dst":random_ip(),
        "proto":row["proto"],
        "service":row["service"],
        "attack":results[i][1],
        "source":results[i][2]
    })

print("\n🧠 SUMMARY:")

attack_logs=[l for l in logs if l["attack"]!="normal"]

if len(attack_logs)>0:
    for l in attack_logs[:3]:
        print(f"{l['attack']} attack from {l['src']} → {l['dst']} via {l['proto']}/{l['service']} (Detected by {l['source']})")
else:
    for l in logs[:3]:
        print(f"Normal traffic from {l['src']} → {l['dst']} via {l['proto']}/{l['service']} (Detected by {l['source']})")

# =========================
# EXPLAINABILITY
# =========================
if len(attack_logs)==0:
    print("\n✔ No attacks detected — system normal")
    print("🚨 THREAT LEVEL: NONE")
    print("🛡️ No action needed")

else:
    src=Counter([l["src"] for l in attack_logs]).most_common(1)[0][0]
    dst=Counter([l["dst"] for l in attack_logs]).most_common(1)[0][0]
    atype=Counter([l["attack"] for l in attack_logs]).most_common(1)[0][0]
    source=Counter([l["source"] for l in attack_logs]).most_common(1)[0][0]

    print("\n🔍 EXPLAINABILITY REPORT:")
    print("WHO:",src)
    print("WHERE:",dst)
    print("WHAT:",atype)
    print("DETECTED BY:",source)

    print("\n🚨 THREAT LEVEL:", "HIGH" if len(attack_logs)>50 else "LOW")

    print("\n🧠 WHY ANALYSIS:")
    print(f"- {atype} detected frequently")
    print(f"- {dst} targeted multiple times")
    print(f"- Detection mainly via {source}")

    print("\n🛡️ RECOMMENDED ACTIONS:")
    print(f"- Block IP {src}")
    print("- Monitor traffic")
    print("- Apply firewall rules")

Epoch 1/3
2740/2740 ━━━━━━━━━━━━━━━━━━━━ 125s 42ms/step - accuracy: 0.9918 - loss: 0.0232
Epoch 2/3
2740/2740 ━━━━━━━━━━━━━━━━━━━━ 99s 36ms/step - accuracy: 0.9942 - loss: 0.0175
Epoch 3/3
2740/2740 ━━━━━━━━━━━━━━━━━━━━ 98s 36ms/step - accuracy: 0.9951 - loss: 0.0143

⏱ TRAINING TIME: 322.58 sec
Loaded Rules: 137
⏱ DETECTION TIME/SEQ: 0.001001688945815027

⚙️ OPTIMIZING RULES...

🔥 OPTIMIZED RULES:
('opt_udp_generic', '\\budp.*generic\\b', 'generic')
('opt_tcp_exploits', '\\btcp.*exploits\\b', 'exploits')
('opt_tcp_fuzzers', '\\btcp.*fuzzers\\b', 'fuzzers')
('opt_tcp_reconnaissance', '\\btcp.*reconnaissance\\b', 'reconnaissance')
('opt_udp_reconnaissance', '\\budp.*reconnaissance\\b', 'reconnaissance')
('opt_udp_fuzzers', '\\budp.*fuzzers\\b', 'fuzzers')
('opt_tcp_dos', '\\btcp.*dos\\b', 'dos')
('opt_tcp_generic', '\\btcp.*generic\\b', 'generic')
('opt_udp_exploits', '\\budp.*exploits\\b', 'exploits')
('opt_tcp_shellcode', '\\btcp.*shellcode\\b', 'shellcode')

CONFUSION MATRIX:
 [[3608

In [5]:
# =========================
# FINAL SOC SYSTEM (ULTIMATE + SELECTIVE OPTIMIZATION)
# =========================

import pandas as pd
import numpy as np
import time
import re
import random
from collections import Counter, defaultdict

# =========================
# LOAD DATA
# =========================
train_df = pd.read_csv("UNSW_NB15_grouped_training_class.csv")
test_df  = pd.read_csv("UNSW_NB15_grouped_testing_class.csv")

train_df["target"] = train_df["4_class"].apply(lambda x: 0 if x == "Benign" else 1)
test_df["target"]  = test_df["4_class"].apply(lambda x: 0 if x == "Benign" else 1)

# =========================
# PREPROCESSING
# =========================
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

drop_cols = ["id","attack_cat","label","4_class","target"]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df["target"]

X_test  = test_df.drop(columns=drop_cols)
y_test  = test_df["target"]

cat_cols = ["proto","service","state"]

preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)],
    remainder="passthrough"
)

X_train = preprocessor.fit_transform(X_train)
X_test  = preprocessor.transform(X_test)

scaler = StandardScaler(with_mean=False)
X_train = scaler.fit_transform(X_train).toarray()
X_test  = scaler.transform(X_test).toarray()

# =========================
# SEQUENCES
# =========================
def create_sequences(X, y, seq_len):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_len):
        X_seq.append(X[i:i+seq_len])
        y_seq.append(1 if np.any(y.iloc[i:i+seq_len].values == 1) else 0)
    return np.array(X_seq), np.array(y_seq)

seq_len = 10
X_train_seq, y_train_seq = create_sequences(X_train, y_train, seq_len)
X_test_seq, y_test_seq   = create_sequences(X_test, y_test, seq_len)

# =========================
# MODEL
# =========================
from tensorflow.keras.layers import Input, Dense, LayerNormalization, MultiHeadAttention, Dropout, GlobalAveragePooling1D
from tensorflow.keras.models import Model

def transformer_block(inputs):
    x = MultiHeadAttention(key_dim=64, num_heads=4)(inputs, inputs)
    x = LayerNormalization()(x + inputs)
    ffn = Dense(128, activation="relu")(x)
    ffn = Dense(inputs.shape[-1])(ffn)
    return LayerNormalization()(x + ffn)

inputs = Input(shape=(seq_len, X_train_seq.shape[2]))
x = transformer_block(inputs)
x = transformer_block(x)
x = GlobalAveragePooling1D()(x)
x = Dense(64, activation="relu")(x)
x = Dropout(0.3)(x)
outputs = Dense(1, activation="sigmoid")(x)

model = Model(inputs, outputs)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# =========================
# TRAIN
# =========================
start_train = time.time()
model.fit(X_train_seq, y_train_seq, epochs=3, batch_size=64)
print("\n⏱ TRAINING TIME:", round(time.time()-start_train,2), "sec")

# =========================
# RULE SYSTEM
# =========================
IMPORTANT_PROTOCOLS = ["tcp","udp","icmp"]
MAX_RULES_PER_ATTACK = 5

rule_stats = defaultdict(lambda: {"count":0,"confidence":0})

def load_rules(filepath):
    rules=[]
    try:
        with open(filepath,"r") as f:
            for line in f:
                line=line.strip()
                if not line or line.startswith("#"): continue
                if line.endswith(","): line=line[:-1]
                try:
                    name,pattern,cat=eval(line)
                    rules.append((name,re.compile(pattern,re.I),cat))
                except:
                    continue
    except:
        open(filepath,"w").close()
    return rules

def rule_exists(rules, pattern):
    return any(p.pattern==pattern for _,p,_ in rules)

def append_rule(filepath,name,pattern,cat):
    with open(filepath,"a") as f:
        f.write(f'("{name}", r"{pattern}", "{cat}"),\n')

rules = load_rules("rules.txt")
print("Loaded Rules:",len(rules))

attack_rule_counter = Counter()

# =========================
# DETECTION
# =========================
texts=[f"{r.proto} {r.service} {r.state} {r.attack_cat}" for _,r in test_df.iterrows()]

start_detect = time.time()
preds = model.predict(X_test_seq, verbose=0)

results=[]

for i in range(len(X_test_seq)):
    row=test_df.iloc[i+seq_len-1]
    text=texts[i+seq_len-1]

    matched=False

    for name,p,cat in rules:
        if p.search(text):
            results.append((1,cat,"RULE"))
            rule_stats[name]["count"]+=1
            matched=True
            break

    if not matched:
        prob=preds[i][0]

        if prob>0.9:
            attack=row["attack_cat"].lower()

            if attack!="normal" and row["proto"] in IMPORTANT_PROTOCOLS:
                if attack_rule_counter[attack]<MAX_RULES_PER_ATTACK:
                    pattern=f"\\b{row['proto']}.*{attack}\\b"

                    if not rule_exists(rules,pattern):
                        name=f"dl_{attack}_{attack_rule_counter[attack]}"
                        append_rule("rules.txt",name,pattern,attack)
                        rules.append((name,re.compile(pattern,re.I),attack))
                        attack_rule_counter[attack]+=1
                        rule_stats[name]["confidence"]=prob

                        print("\n🆕 RULE ADDED:",name)

            results.append((1,attack,"DL"))
        else:
            results.append((0,"normal","DL"))

print("⏱ DETECTION TIME/SEQ:",(time.time()-start_detect)/len(X_test_seq))

# =========================
# RULE OPTIMIZER
# =========================
print("\n⚙️ OPTIMIZING RULES...")

# track rules used
optimized_input_names = set([r[0] for r in rules])

# remove weak
rules = [r for r in rules if rule_stats[r[0]]["count"]>5 or rule_stats[r[0]]["confidence"]>0.9]

# rank
ranked_rules = sorted(rules,key=lambda r:rule_stats[r[0]]["count"],reverse=True)

# generalize
def optimize_rules(rules):
    grouped = defaultdict(list)
    for name,pattern,cat in rules:
        proto = re.search(r"\\b(\w+)",pattern.pattern)
        if proto:
            grouped[(proto.group(1),cat)].append(pattern.pattern)

    optimized=[]
    for (proto,cat),patterns in grouped.items():
        optimized.append((f"opt_{proto}_{cat}",f"\\b{proto}.*{cat}\\b",cat))
    return optimized

optimized_rules = optimize_rules(ranked_rules)

# =========================
# SELECTIVE REPLACEMENT
# =========================
def replace_optimized_rules(filepath, optimized_rules, old_rule_names):

    new_lines=[]

    with open(filepath,"r") as f:
        for line in f:
            keep=True
            for name in old_rule_names:
                if f'("{name}"' in line:
                    keep=False
                    break
            if keep:
                new_lines.append(line)

    for name,pattern,cat in optimized_rules:
        new_lines.append(f'("{name}", r"{pattern}", "{cat}"),\n')

    with open(filepath,"w") as f:
        f.writelines(new_lines)

replace_optimized_rules("rules.txt",optimized_rules,optimized_input_names)

print("✅ Optimized rules selectively updated")

# =========================
# METRICS
# =========================
from sklearn.metrics import confusion_matrix,precision_score,recall_score,f1_score,accuracy_score

y_pred=[x[0] for x in results]

cm=confusion_matrix(y_test_seq,y_pred)
tn,fp,fn,tp=cm.ravel()

print("\nCONFUSION MATRIX:\n",cm)
print("\nTP:",tp,"TN:",tn,"FP:",fp,"FN:",fn)

print("Accuracy:",accuracy_score(y_test_seq,y_pred))
print("Precision:",precision_score(y_test_seq,y_pred))
print("Recall:",recall_score(y_test_seq,y_pred))
print("F1:",f1_score(y_test_seq,y_pred))

print("\nTotal Attacks:",sum(y_pred))

# =========================
# SOC OUTPUT
# =========================
def random_ip():
    return ".".join(str(random.randint(1,255)) for _ in range(4))

logs=[]
for i in range(len(results)):
    row=test_df.iloc[i]
    logs.append({
        "src":random_ip(),
        "dst":random_ip(),
        "proto":row["proto"],
        "service":row["service"],
        "attack":results[i][1],
        "source":results[i][2]
    })

print("\n🧠 SUMMARY:")

attack_logs=[l for l in logs if l["attack"]!="normal"]

if attack_logs:
    for l in attack_logs[:3]:
        print(f"{l['attack']} attack from {l['src']} → {l['dst']} via {l['proto']}/{l['service']} ({l['source']})")
else:
    for l in logs[:3]:
        print(f"Normal traffic from {l['src']} → {l['dst']} via {l['proto']}/{l['service']} ({l['source']})")

if not attack_logs:
    print("\n✔ No attacks detected — system normal")
else:
    src=Counter([l["src"] for l in attack_logs]).most_common(1)[0][0]
    dst=Counter([l["dst"] for l in attack_logs]).most_common(1)[0][0]
    atype=Counter([l["attack"] for l in attack_logs]).most_common(1)[0][0]

    print("\n🔍 WHO:",src)
    print("WHERE:",dst)
    print("WHAT:",atype)

    print("\n🚨 THREAT LEVEL:", "HIGH" if len(attack_logs)>50 else "LOW")

    print("\n🧠 WHY:")
    print(f"{atype} frequent attack detected")

    print("\n🛡️ ACTION:")
    print(f"Block IP {src}")

Epoch 1/3
2740/2740 ━━━━━━━━━━━━━━━━━━━━ 87s 30ms/step - accuracy: 0.9913 - loss: 0.0248
Epoch 2/3
2740/2740 ━━━━━━━━━━━━━━━━━━━━ 100s 36ms/step - accuracy: 0.9938 - loss: 0.0183
Epoch 3/3
2740/2740 ━━━━━━━━━━━━━━━━━━━━ 100s 36ms/step - accuracy: 0.9947 - loss: 0.0161

⏱ TRAINING TIME: 287.9 sec
Loaded Rules: 137
⏱ DETECTION TIME/SEQ: 0.00038780746363576373

⚙️ OPTIMIZING RULES...
✅ Optimized rules selectively updated

CONFUSION MATRIX:
 [[36755   217]
 [   18 45332]]

TP: 45332 TN: 36755 FP: 217 FN: 18
Accuracy: 0.9971453560409125
Precision: 0.9952358998002152
Recall: 0.9996030871003307
F1: 0.9974147130331467

Total Attacks: 45549

🧠 SUMMARY:
reconnaissance attack from 135.85.28.145 → 164.191.160.210 via tcp/- (DL)
malware attack from 174.237.139.85 → 129.2.220.82 via tcp/- (RULE)
dos attack from 215.98.178.95 → 210.115.233.182 via igmp/- (DL)

🔍 WHO: 135.85.28.145
WHERE: 164.191.160.210
WHAT: generic

🚨 THREAT LEVEL: HIGH

🧠 WHY:
generic frequent attack detected

🛡️ ACTION:
Block IP 